# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [1]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

In [2]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [3]:
# 🤖 AGENT FUNCTION (TO IMPLEMENT)

import re

def agent(query: str):
    try:
        if not isinstance(query, str) or query.strip() == "":
            return {"type": "error", "result": "Empty or invalid query."}

        query_lower = query.lower()

        if "calculate" in query_lower:
            expression = query_lower.split("calculate", 1)[1]
            expression = re.sub(r"[^0-9+\-*/().% ]", "", expression).strip()

            if not expression:
                return {"type": "error", "result": "No mathematical expression found.Write expression after 'calculate'"}

            calc_result = calculator(expression)

            if calc_result == "Error in calculation":
                return {"type": "error", "result": f"Invalid mathematical expression"}

            try:
                numeric_result = float(calc_result)
                if numeric_result.is_integer():
                    numeric_result = int(numeric_result)
                return {"type": "calculation", "result": numeric_result}
            except ValueError:
                return {"type": "calculation", "result": calc_result}

        elif "keywords" in query_lower:
            keywords = extract_keywords(query)
            return {"type": "keywords", "result": keywords}

        else:
            return {
                "type": "general",
                "result": "I can help with calculations or keyword extraction. "
                           "Try including the word 'calculate' or 'keywords' in your query.",
            }

    except Exception as e:
        return {"type": "error", "result": f"Unexpected error: {str(e)}"}

## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [4]:
# 🧪 Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    "Calculate 10 / 0",              
    "Calculate two plus two",       
    ""                                
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': 25}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['keywords', 'extract', 'transforming', 'industries', 'intelligence']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': "I can help with calculations or keyword extraction. Try including the word 'calculate' or 'keywords' in your query."}
--------------------------------------------------
Query: Calculate 10 / 0
Response: {'type': 'error', 'result': 'Invalid mathematical expression'}
--------------------------------------------------
Query: Calculate two plus two
Response: {'type': 'error', 'result': "No mathematical expression found.Write expression after 'calculate'"}
--------------------------------------------------
Query: 
Response: {'type': 'error', 'result': 'Empty 

In [5]:
# 🎯 Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))

**Added a new tool - word counter - that counts the total words in query**

In [6]:
def word_counter(text: str) -> dict:
    try:
        words = text.split()
        return {"word_count": len(words), "char_count": len(text)}
    except Exception:
        return {"word_count": 0, "char_count": 0}


**Added a new tool - text reverser - that reverses the query**

In [7]:
def text_reverser(text: str) -> str:
    try:
        return text[::-1]
    except Exception:
        return ""

**Added loggs to the output**

In [8]:
import logging
import time

logger = logging.getLogger("agent_pipeline")
logger.setLevel(logging.INFO)

if not logger.handlers:
    _handler = logging.StreamHandler()
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", datefmt="%H:%M:%S"))
    logger.addHandler(_handler)

execution_log = []


**Implementing query logic for all the tools combined**

In [11]:
def _route_query(query: str):
    if not isinstance(query, str) or query.strip() == "":
        logger.warning("Received empty or invalid query.")
        return {"type": "error", "result": "Empty or invalid query received."}

    query_lower = query.lower()


    if "calculate" in query_lower:
        expression = query_lower.split("calculate", 1)[1]
        expression = re.sub(r"[^0-9+\-*/().% ]", "", expression).strip()

        if not expression:
            logger.error(f"No expression found in query: '{query}'")
            return {"type": "error", "result": "No mathematical expression found after 'calculate'."}

        calc_result = calculator(expression)
        if calc_result == "Error in calculation":
            logger.error(f"Calculator failed on expression: '{expression}'")
            return {"type": "error", "result": f"Invalid mathematical expression: '{expression}'"}

        try:
            numeric_result = float(calc_result)
            if numeric_result.is_integer():
                numeric_result = int(numeric_result)
            logger.info(f"Routed to calculator | query='{query}' -> {numeric_result}")
            return {"type": "calculation", "result": numeric_result}
        except ValueError:
            logger.info(f"Routed to calculator | query='{query}' -> {calc_result}")
            return {"type": "calculation", "result": calc_result}


    elif "keywords" in query_lower:
        keywords = extract_keywords(query)
        logger.info(f"Routed to keyword extractor | query='{query}' -> {keywords}")
        return {"type": "keywords", "result": keywords}


    elif "count" in query_lower:
        counts = word_counter(query)
        logger.info(f"Routed to word counter | query='{query}' -> {counts}")
        return {"type": "word_count", "result": counts}


    elif "reverse" in query_lower:
        reversed_text = text_reverser(query)
        logger.info(f"Routed to text reverser | query='{query}' -> {reversed_text}")
        return {"type": "reversed_text", "result": reversed_text}


    else:
        logger.info(f"No tool matched | query='{query}' -> general fallback")
        return {
            "type": "general",
            "result": "I can help with calculations, keyword extraction, word counts, or "
                       "reversing text. Try including a word like 'calculate', 'keywords', "
                       "'count', or 'reverse'.",
        }


def agent_v2(query: str):
    try:
        response = _route_query(query)
    except Exception as e:
        logger.exception(f"Unexpected error while handling query: '{query}'")
        response = {"type": "error", "result": f"Unexpected error: {str(e)}"}

    execution_log.append({
        "timestamp": time.strftime("%H:%M:%S"),
        "query": query,
        "type": response["type"],
    })
    return response


**clears the log before every session**

**Final query checks for all the tools**

In [15]:
execution_log.clear() 

bonus_queries = [
    "Calculate 8 * 9",
    "Calculate 100 / 4",
    "Extract keywords from this sentence about renewable energy policy",
    "Count words in this sentence please",   
    "Reverse this text",                      
    "Tell me a joke",                         
]

for q in bonus_queries:
    print("Query:", q)
    print("Response:", agent_v2(q))

print()
print("Execution Log (audit trail)")
for entry in execution_log:
    print(entry)


09:17:41 | INFO | Routed to calculator | query='Calculate 8 * 9' -> 72
09:17:41 | INFO | Routed to calculator | query='Calculate 100 / 4' -> 25
09:17:41 | INFO | Routed to keyword extractor | query='Extract keywords from this sentence about renewable energy policy' -> ['keywords', 'extract', 'sentence', 'policy', 'about']
09:17:41 | INFO | Routed to word counter | query='Count words in this sentence please' -> {'word_count': 6, 'char_count': 35}
09:17:41 | INFO | Routed to text reverser | query='Reverse this text' -> txet siht esreveR
09:17:41 | INFO | No tool matched | query='Tell me a joke' -> general fallback


Query: Calculate 8 * 9
Response: {'type': 'calculation', 'result': 72}
Query: Calculate 100 / 4
Response: {'type': 'calculation', 'result': 25}
Query: Extract keywords from this sentence about renewable energy policy
Response: {'type': 'keywords', 'result': ['keywords', 'extract', 'sentence', 'policy', 'about']}
Query: Count words in this sentence please
Response: {'type': 'word_count', 'result': {'word_count': 6, 'char_count': 35}}
Query: Reverse this text
Response: {'type': 'reversed_text', 'result': 'txet siht esreveR'}
Query: Tell me a joke
Response: {'type': 'general', 'result': "I can help with calculations, keyword extraction, word counts, or reversing text. Try including a word like 'calculate', 'keywords', 'count', or 'reverse'."}

Execution Log (audit trail)
{'timestamp': '09:17:41', 'query': 'Calculate 8 * 9', 'type': 'calculation'}
{'timestamp': '09:17:41', 'query': 'Calculate 100 / 4', 'type': 'calculation'}
{'timestamp': '09:17:41', 'query': 'Extract keywords from this se

## Key Observations

- Routing on plain substring checks ("calculate", "keywords") is fast and simple, but has a known edge case. A word like *"recalculate"* contains "calculate" as a substring, so it gets misrouted to the calculator instead of falling back to general.


- The provided calculator tool uses raw eval() and swallows every exception into one generic string, "Error in calculation" - so the agent can label something as a failure but can't distinguish *why* it failed (bad syntax vs. division by zero) without extra logic.


- The provided extract_keywords tool ranks purely by word length (>4 chars) with no stopword filtering, so instruction words like *"extract"* or *"keywords"* can show up in its own output - a clear example of a tool's limitations surfacing through the pipeline it's part of.


- Wrapping every route in the same {"type", "result"} JSON shape, plus a top-level try/except safety net, meant every test case - including empty strings and malformed expressions - returned a clean labeled response instead of crashing.


- The bonus features (logging, word counter, text reverser) were added without touching the required agent() function at all - they live in a separate agent_v2, showing the router pattern scales cleanly without risking the graded core.

## Conclusion

This project implements a single-agent router that reads a query, matches it against simple conditional patterns, and dispatches it to the correct tool - calculator, keyword extractor, or a general fallback - always returning a consistent, structured JSON response. Error handling at both the tool level and the top level of agent() ensures the pipeline degrades gracefully rather than failing outright on bad input.

Beyond the core requirement, the bonus section layers in logging and two additional tools to show how a rule-based router can be extended without disturbing existing behavior. The main takeaway is that substring-based routing is easy to reason about and debug, but it's inherently brittle - real-world agents typically need either stricter pattern matching (e.g., word boundaries) or intent classification to avoid the kind of false positives seen with queries like "recalculate."